# 此文档用于实现更高级的网络结构
* CNN,RNN以及Transformer
## 1.CNN的实现
* 对于卷积的理解
> 信号分析中，卷积用于求冲激响应的累计效果
> 概率论中，卷积用于求线密度（想象一下一条线平移过平面，故命名为卷积）
> 图像处理中，卷积用于求卷积算子和图像数据点积（卷积算子滑过图像的过程类似于上面线滑过平面的过程
* 1*1的卷积核主要用于将变换图像的通道数

In [1]:
import torch as T

# 1*1卷积核实现
def corr2d_one(X,K):
    c_i,h,w = X.shape
    c_o = K.shape[0]
    X = X.reshape(c_i,h*w) # 相当于Flatten
    K = K.reshape(c_o,c_i)
    Y = T.matmul(K,X)
    return Y.reshape(c_o,h,w)

X = T.normal(0,1,size=(3,3,3))
K = T.normal(0,1,size=(2,3,1,1))

print(corr2d_one(X,K))

tensor([[[-0.6423, -0.4386,  0.5380],
         [ 0.2455, -1.3318, -1.0462],
         [-0.2739, -0.3180, -0.1758]],

        [[ 0.0648, -0.6529, -0.5075],
         [ 0.4567,  0.4224, -0.4576],
         [-0.0172, -0.1514, -0.3306]]])


* Lenet实现

In [13]:
import torch as T
import torchvision as tv
import torch.nn as nn
import torch.nn.init as init

from torch.utils import data
from torchvision import transforms

# 检查是否有可用的 GPU
device = T.device('cuda' if T.cuda.is_available() else 'cpu')

# 定义数据生成器
def load_data(batch_size):
    # 将图像从 PIL 格式或 NumPy 数组转换为 PyTorch 的张量（tensor）
    trans = [transforms.ToTensor()]
    # 是一个用于将多个转换操作组合在一起的工具
    trans = transforms.Compose(trans)
    # 加载mnist数据集
    train_data = tv.datasets.MNIST(root='./data', train=True, download=True, transform=trans)
    test_data = tv.datasets.MNIST(root='./data', train=False, download=True, transform=trans)
    # 定义数据加载器
    train_iter = data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_iter = data.DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_iter, test_iter

# 定义网络并移动到 GPU
net = nn.Sequential(
    nn.Conv2d(1,6,kernel_size=(5,5),padding=(2,2)),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(2,2),stride=(2,2)),
    nn.Conv2d(6,16,kernel_size=(5,5)),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(2,2),stride=(2,2)),
    nn.Flatten(),
    nn.Linear(16*5*5,120),
    nn.ReLU(),
    nn.Linear(120,84),
    nn.ReLU(),
    nn.Linear(84,10)
).to(device)

# 初始化函数
def initialize_weights(model):
    for layer in model:
        if isinstance(layer, nn.Linear) or isinstance(layer, nn.Conv2d):
            init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # 使用 He 初始化
            if layer.bias is not None:
                init.zeros_(layer.bias)  # 偏置初始化为 0

# 调用初始化函数
initialize_weights(net)

# 损失函数
loss = nn.CrossEntropyLoss().to(device)  # 将损失函数移动到 GPU

# 优化器
sgd = T.optim.SGD(net.parameters(), lr=0.1)

# 训练
epoch = 10
batch_size = 256
train_iter, test_iter = load_data(batch_size)
for i in range(epoch):
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)  # 将数据移动到 GPU
        sgd.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.backward()
        sgd.step()
    
    # 测试阶段
    net.eval()  # 设置模型为评估模式
    correct = 0
    total = 0
    test_loss = 0.0

    with T.no_grad():
        for X, y in test_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            test_loss += loss(y_hat, y).item()  # 累加测试损失
            predicted = T.argmax(y_hat, dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    accuracy = correct / total
    print(f"Epoch: {i + 1}, Loss: {test_loss / len(test_iter):.4f}, Test Accuracy: {accuracy:.4f}")


Epoch: 1, Loss: 0.2623, Test Accuracy: 0.9133
Epoch: 2, Loss: 0.1121, Test Accuracy: 0.9652
Epoch: 3, Loss: 0.1242, Test Accuracy: 0.9552
Epoch: 4, Loss: 0.0722, Test Accuracy: 0.9764
Epoch: 5, Loss: 0.0616, Test Accuracy: 0.9797
Epoch: 6, Loss: 0.0516, Test Accuracy: 0.9826
Epoch: 7, Loss: 0.0741, Test Accuracy: 0.9765
Epoch: 8, Loss: 0.0449, Test Accuracy: 0.9855
Epoch: 9, Loss: 0.0868, Test Accuracy: 0.9704
Epoch: 10, Loss: 0.0432, Test Accuracy: 0.9858
